In [1]:
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt

RAW_PATH = '../02_Data/'
PROCESSED_PATH = '../02_Data/processed'
os.makedirs(PROCESSED_PATH, exist_ok=True)

def clean_price(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r'[\$,€£]', '', regex=True).str.strip() , errors='coerce'
    )

def snapshot(df, name):
    return {
        'dataset' : name,
        'rows' : df.shape[0],
        'cols': df.shape[1],
        'columns': list(df.columns),
        'dtypes': df.dtypes.astype(str).to_dict()
    }


In [11]:
reviews_raw = pd.read_csv(os.path.join(RAW_PATH, 'reviews.csv'))
reviews = reviews_raw.copy()

# Dates
reviews['date'] = pd.to_datetime(reviews['date'], errors='coerce')
print(f"Invalid review dates: {reviews['date'].isnull().sum()}")

# IDs
reviews['id'] = pd.to_numeric(reviews['id'], errors='coerce').astype('Int64')
reviews['listing_id'] = pd.to_numeric(reviews['listing_id'], errors='coerce').astype('Int64')

# Duplicate review IDs
dup_ids = reviews.duplicated(subset=['id'], keep=False).sum()
exact_dupes = reviews.duplicated().sum()
print(f"Duplicate review IDs: {dup_ids}")
print(f"Exact duplicate rows: {exact_dupes}")


# Missing listing IDs
print(f"Missing listing_ids: {reviews['listing_id'].isnull().sum()}")

# FK integrity
listings_ids = set(listings['id'])
review_orphans = set(reviews['listing_id'].dropna()) - listings_ids
print(f"Review listing_ids NOT in listings: {len(review_orphans):,}")

# Preserve review text (comments) — DO NOT impute
if 'comments' in reviews.columns:
    print(f"Review text present in {reviews['comments'].notna().sum():,} rows")
    print("Text preserved; will NOT be invented or imputed")

Invalid review dates: 0
Duplicate review IDs: 0
Exact duplicate rows: 0
Missing listing_ids: 0
Review listing_ids NOT in listings: 77
Review text present in 1,033,409 rows
Text preserved; will NOT be invented or imputed


In [9]:
reviews.to_csv(os.path.join(PROCESSED_PATH, 'reviews_clean.csv'), index=False)
print(f"Saved reviews_clean.csv  ({reviews.shape[0]:,} rows)")

Saved reviews_clean.csv  (1,033,523 rows)
